# Parameter Studies, Multi-output Redshifts, and Galaxy Trees

This notebook covers:
- Varying model parameters with `input_overrides`
- Running grids of parameter values
- Outputting multiple redshifts in a single GALFORM run
- Building galaxy and halo merger trees

In [ ]:
from galform_execution.submit_galform_job import GalformSubmitter, _default_galform_dir

GALFORM_DIR = _default_galform_dir()

## 1. Overriding model parameters with `input_overrides`

`input_overrides` injects arbitrary `name → value` pairs into the GALFORM parameter file via `replace_variable.csh`, after the model's own `extra_replacements`. The parameter name must match a variable in the base `.input.ref` file exactly.

Use this for one-off tweaks or exploratory runs without creating a new entry in `models.json`.

In [ ]:
s = GalformSubmitter(
    GALFORM_DIR,
    nbody_sim="Mill2", model="lc16",
    iz=40, nvol="1-64",
    input_overrides={
        "vhotdisk":  "290",
        "vhotburst": "290",
    },
    output_folder_name="Mill2_vhot290",
)

# Verify the injected lines in the generated tcsh script
script = s._create_tcsh_script(iz=40)
for line in script.splitlines():
    if "vhotdisk" in line or "vhotburst" in line:
        print(line)

## 2. Parameter grid — single parameter

Loop over values to generate one submitter (and one set of SLURM jobs) per configuration.

In [ ]:
vhot_values = [200, 250, 290, 350]

grid = [
    GalformSubmitter(
        GALFORM_DIR,
        nbody_sim="Mill2", model="lc16",
        iz=40, nvol="1-64",
        input_overrides={"vhotdisk": str(v), "vhotburst": str(v)},
        output_folder_name=f"PS_vhot{v}",
    )
    for v in vhot_values
]

for s in grid:
    print(f"vhot={s.input_overrides['vhotdisk']:>3}  →  {s.models_dir}")

## 3. Parameter grid — two parameters

In [ ]:
vhot_values  = [250, 290]
alpha_values = ["0.6", "0.8", "1.0"]

grid_2d = [
    GalformSubmitter(
        GALFORM_DIR,
        nbody_sim="Mill2", model="lc16",
        iz=40, nvol="1-64",
        input_overrides={"vhotdisk": str(v), "vhotburst": str(v), "alpha_cool": alpha},
        output_folder_name=f"PS_vhot{v}_alpha{alpha.replace('.', 'p')}",
    )
    for v in vhot_values
    for alpha in alpha_values
]

print(f"{len(grid_2d)} configurations:")
for s in grid_2d:
    print(f"  {s.output_folder_name}")

In [ ]:
# Submit the entire grid
all_job_ids = []
for s in grid_2d:
    job_ids = s.submit_all_jobs(dry_run=False)
    print(f"{s.output_folder_name}: {job_ids}")
    all_job_ids.extend(job_ids)
print(f"\nTotal SLURM jobs submitted: {len(all_job_ids)}")

## 4. Multi-output redshifts

By default each job outputs at a single redshift (`nout=1`). You can request multiple outputs per run in two ways:

- **`output_iz_list`** — list of snapshot indices; redshifts are resolved from the simulation's snapshot file
- **`output_redshifts`** — list of explicit redshift values

Both set `nout` and populate `zout` in the GALFORM parameter file.

> **`iz` vs `output_iz_list`**: `iz` is the snapshot the SLURM job runs *from*; `output_iz_list` controls what redshifts are *written to disk*.

In [ ]:
# Output at four snapshots in one run, redshifts resolved from Mill2.txt
s_multi = GalformSubmitter(
    GALFORM_DIR,
    nbody_sim="Mill2", model="lc16",
    iz=67, nvol="1-64",
    output_iz_list=[20, 30, 40, 67],
    output_folder_name="Mill2_MultiOut",
)

script = s_multi._create_tcsh_script(iz=67)
for line in script.splitlines():
    if "nout" in line or "zout" in line:
        print(line)

In [ ]:
# Or specify redshifts directly
s_multi_z = GalformSubmitter(
    GALFORM_DIR,
    nbody_sim="Mill2", model="lc16",
    iz=67, nvol="1-64",
    output_redshifts=[0.0, 0.5, 1.0, 2.0, 3.0],
    output_folder_name="Mill2_MultiZ",
)

script_z = s_multi_z._create_tcsh_script(iz=67)
for line in script_z.splitlines():
    if "nout" in line or "zout" in line:
        print(line)

## 5. Building galaxy and halo merger trees

Set `build_galaxy_trees = .true.` via `input_overrides`. When this is combined with multiple output snapshots, `GalformSubmitter` **automatically injects** `mgalmin_output_descendants = .true.` to track descendant links across outputs — you do not need to set this manually.

Add `output_halo_trees = .true.` to also write halo merger trees.

In [ ]:
s_trees = GalformSubmitter(
    GALFORM_DIR,
    nbody_sim="Mill2", model="lc16",
    iz=67, nvol="1-64",
    output_iz_list=[20, 40, 67],
    input_overrides={
        "build_galaxy_trees": ".true.",
        "output_halo_trees":  ".true.",
    },
    output_folder_name="Mill2_Trees",
)

# mgalmin_output_descendants injected automatically
print("input_overrides:")
for k, v in s_trees.input_overrides.items():
    print(f"  {k} = {v}")

In [ ]:
# Verify tree flags appear in the generated tcsh script
script_trees = s_trees._create_tcsh_script(iz=67)
for line in script_trees.splitlines():
    if any(k in line for k in ["build_galaxy_trees", "output_halo_trees", "mgalmin_output"]):
        print(line)

In [ ]:
s_trees.submit_all_jobs(dry_run=True)

## 6. Trees on a large simulation

The same pattern scales to L800 (1024 subvolumes).

In [ ]:
s_l800_trees = GalformSubmitter(
    GALFORM_DIR,
    nbody_sim="L800", model="lc16.newmg",
    iz=271, nvol="1-1024",
    output_iz_list=[82, 105, 155, 207, 271],
    input_overrides={"build_galaxy_trees": ".true."},
    output_folder_name="L800_Trees",
    partition="cosma8",  # 128 CPUs per node
)

print(f"Subvolumes       : {s_l800_trees.nvol_count}")
print(f"Output snapshots : {s_l800_trees.output_iz_list}")
print(f"input_overrides  : {s_l800_trees.input_overrides}")

# On cosma8 (128 CPUs), 1024 ivols → 128 workers each running 8 ivols sequentially
script = s_l800_trees.create_job_script(iz=271, tcsh_path="/tmp/l800.csh")
for line in script.splitlines():
    if "cpus-per-task" in line or "seq 1" in line or "task_id -le" in line:
        print(line)